# C11-neural-training — Practice p10 — Solution


**Type:** constrained coding · **Difficulty:** core · **Concepts:** dropout


One module is switched between modes. Resetting the seed before replay restores
the same random-number sequence; evaluation uses the identity map.


In [ ]:
import torch
import torch.nn as nn

def dropout_train_eval(x, p, seed=20260804):
    values = x.detach().clone().to(dtype=torch.float64, device="cpu")
    drop = nn.Dropout(p=p)
    drop.train()
    torch.manual_seed(seed)
    first, second = drop(values), drop(values)
    torch.manual_seed(seed)
    repeat_first, repeat_second = drop(values), drop(values)
    drop.eval()
    eval_first, eval_second = drop(values), drop(values)
    return {k: v.detach().clone() for k, v in {
        "train_first": first, "train_second": second, "repeat_first": repeat_first,
        "repeat_second": repeat_second, "eval_first": eval_first, "eval_second": eval_second}.items()}

x_p10 = torch.arange(1, 17, dtype=torch.float64).reshape(4, 4)
result_p10 = dropout_train_eval(x_p10, 0.25)


### Answer check


In [ ]:
for key in ("train_first", "train_second"):
    ratio = result_p10[key] / x_p10
    assert torch.all((ratio == 0.0) | torch.isclose(ratio, torch.tensor(4/3, dtype=torch.float64), atol=1e-12, rtol=1e-12))
assert torch.equal(result_p10["train_first"], result_p10["repeat_first"])
assert torch.equal(result_p10["train_second"], result_p10["repeat_second"])
assert not torch.equal(result_p10["train_first"], result_p10["train_second"])
assert torch.allclose(result_p10["eval_first"], x_p10, atol=1e-12, rtol=1e-12)
assert torch.allclose(result_p10["eval_second"], x_p10, atol=1e-12, rtol=1e-12)
